# Hyperparameter Tuning Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: Grid Search from Scratch

The code in `code/tuning.py` implements grid search, random search, and a simple Bayesian optimizer from scratch.

In [ ]:
```python

def grid_search(model_fn, param_grid, X_train, y_train, X_val, y_val):

    keys = list(param_grid.keys())

    values = list(param_grid.values())

    best_score = -float("inf")

    best_params = None

    n_evals = 0

    for combo in itertools.product(*values):

        params = dict(zip(keys, combo))

        model = model_fn(**params)

        model.fit(X_train, y_train)

        score = evaluate(model, X_val, y_val)

        n_evals += 1

        if score > best_score:

            best_score = score

            best_params = params

    return best_params, best_score, n_evals

In [ ]:
```

### Step 2: Random Search from Scratch

In [ ]:
```python

def random_search(model_fn, param_distributions, X_train, y_train,

                  X_val, y_val, n_iter=50, seed=42):

    rng = np.random.RandomState(seed)

    best_score = -float("inf")

    best_params = None

    for _ in range(n_iter):

        params = {k: sample(v, rng) for k, v in param_distributions.items()}

        model = model_fn(**params)

        model.fit(X_train, y_train)

        score = evaluate(model, X_val, y_val)

        if score > best_score:

            best_score = score

            best_params = params

    return best_params, best_score, n_iter

In [ ]:
```

### Step 3: Bayesian Optimization (Simplified)

The core idea: fit a Gaussian process to observed (hyperparameter, score) pairs, then use an acquisition function to decide where to look next.

In [ ]:
```python

class SimpleBayesianOptimizer:

    def __init__(self, search_space, n_initial=5):

        self.search_space = search_space

        self.n_initial = n_initial

        self.X_observed = []

        self.y_observed = []

    def _kernel(self, x1, x2, length_scale=1.0):

        dists = np.sum((x1[:, None, :] - x2[None, :, :]) ** 2, axis=2)

        return np.exp(-0.5 * dists / length_scale ** 2)

    def _fit_gp(self, X_new):

        X_obs = np.array(self.X_observed)

        y_obs = np.array(self.y_observed)

        y_mean = y_obs.mean()

        y_centered = y_obs - y_mean

        K = self._kernel(X_obs, X_obs) + 1e-4 * np.eye(len(X_obs))

        K_star = self._kernel(X_new, X_obs)

        L = np.linalg.cholesky(K)

        alpha = np.linalg.solve(L.T, np.linalg.solve(L, y_centered))

        mu = K_star @ alpha + y_mean

        v = np.linalg.solve(L, K_star.T)

        var = 1.0 - np.sum(v ** 2, axis=0)

        var = np.maximum(var, 1e-6)

        return mu, var

    def _expected_improvement(self, mu, var, best_y):

        sigma = np.sqrt(var)

        z = (mu - best_y) / (sigma + 1e-10)

        ei = sigma * (z * norm_cdf(z) + norm_pdf(z))

        return ei

    def suggest(self):

        if len(self.X_observed) < self.n_initial:

            return sample_random(self.search_space)

        candidates = [sample_random(self.search_space) for _ in range(500)]

        X_cand = np.array([to_vector(c) for c in candidates])

        mu, var = self._fit_gp(X_cand)

        ei = self._expected_improvement(mu, var, max(self.y_observed))

        return candidates[np.argmax(ei)]

    def observe(self, params, score):

        self.X_observed.append(to_vector(params))

        self.y_observed.append(score)

In [ ]:
```

The GP surrogate gives two things at each candidate point: a predicted score (mu) and an uncertainty (var). Expected Improvement balances these: it favors points where the model predicts high scores OR where uncertainty is high. Early on, most points have high uncertainty so the optimizer explores. Later, it focuses on the most promising region.

### Step 4: Compare All Methods

Run all three methods on the same synthetic objective and compare. This comparison uses a simplified wrapper that calls each optimizer with a direct objective function (no model training), so the API differs from the model-based implementations above:

In [ ]:
```python

def synthetic_objective(params):

    lr = params["learning_rate"]

    depth = params["max_depth"]

    return -(np.log10(lr) + 2) ** 2 - (depth - 4) ** 2 + 10

param_grid = {

    "learning_rate": [0.001, 0.01, 0.1, 1.0],

    "max_depth": [2, 3, 4, 5, 6, 7, 8],

}

grid_best = None

grid_score = -float("inf")

grid_history = []

for combo in itertools.product(*param_grid.values()):

    params = dict(zip(param_grid.keys(), combo))

    score = synthetic_objective(params)

    grid_history.append((params, score))

    if score > grid_score:

        grid_score = score

        grid_best = params

param_dist = {

    "learning_rate": ("log_float", 0.001, 1.0),

    "max_depth": ("int", 2, 8),

}

rand_best = None

rand_score = -float("inf")

rand_history = []

rng = np.random.RandomState(42)

for _ in range(28):

    params = {k: sample(v, rng) for k, v in param_dist.items()}

    score = synthetic_objective(params)

    rand_history.append((params, score))

    if score > rand_score:

        rand_score = score

        rand_best = params

optimizer = SimpleBayesianOptimizer(param_dist, n_initial=5)

bayes_history = []

for _ in range(28):

    params = optimizer.suggest()

    score = synthetic_objective(params)

    optimizer.observe(params, score)

    bayes_history.append((params, score))

bayes_score = max(s for _, s in bayes_history)

print(f"{'Method':<20} {'Best Score':>12} {'Evaluations':>12}")

print("-" * 50)

print(f"{'Grid Search':<20} {grid_score:>12.4f} {len(grid_history):>12}")

print(f"{'Random Search':<20} {rand_score:>12.4f} {len(rand_history):>12}")

print(f"{'Bayesian Opt':<20} {bayes_score:>12.4f} {len(bayes_history):>12}")

In [ ]:
```

With the same budget, Bayesian optimization usually finds the best score fastest because it does not waste evaluations in clearly bad regions. Random search covers more ground than grid search. Grid search only wins when you have very few hyperparameters and can afford to be exhaustive.

## Exercises

In [ ]:
1. Run grid search and random search with the same total budget (e.g., 50 evaluations). Compare the best scores found. Run the experiment 10 times with different seeds. How often does random search win?

2. Implement Hyperband from scratch. Start with 81 configurations, each trained for 1 epoch. Keep the top 1/3 at each round and triple their budget. Compare total compute (sum of all epochs across all configs) to running 81 configs for the full budget.

3. Add a learning rate scheduler (cosine annealing) to the gradient boosting implementation from Lesson 11. Does it help compared to a fixed learning rate?

4. Use Optuna to tune a RandomForestClassifier on a real dataset (e.g., sklearn's breast cancer dataset). Use `optuna.visualization.plot_param_importances(study)` to see which hyperparameters matter most. Does it match the importance ranking from this lesson?

5. Implement a simple acquisition function (Expected Improvement) and demonstrate exploration vs exploitation. Plot the surrogate model's mean and uncertainty, and show where EI chooses to evaluate next.